# PyHF Tutorial: Hartree-Fock Calculations in Python

This notebook demonstrates the capabilities of PyHF, a Python library for performing Hartree-Fock quantum chemistry calculations.

## Contents
1. [Basic H₂ Calculation](#1-basic-h₂-calculation)
2. [Energy Analysis and Decomposition](#2-energy-analysis-and-decomposition)
3. [Population Analysis](#3-population-analysis)
4. [Molecular Orbital Analysis](#4-molecular-orbital-analysis)
5. [DIIS Convergence Acceleration](#5-diis-convergence-acceleration)
6. [Potential Energy Surface Scan](#6-potential-energy-surface-scan)
7. [Comparing Different Molecules](#7-comparing-different-molecules)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyhf.molecule import Atom, Molecule
from pyhf.integrals import compute_integrals
from pyhf.scf import RHF
from pyhf.analysis import (
    print_energy_decomposition,
    MullikenPopulation,
    print_koopmans_analysis
)
from pyhf.analysis.energy import decompose_energy, virial_ratio
from pyhf.analysis.koopmans import ionization_energy, electron_affinity, homo_lumo_gap
from pyhf.utils.constants import HARTREE_TO_EV, BOHR_TO_ANGSTROM

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Basic H₂ Calculation

Let's start with a simple hydrogen molecule calculation using Restricted Hartree-Fock (RHF).

In [ ]:
# Create H2 molecule with bond length of 1.4 Bohr (~0.74 Angstrom)
h2 = Molecule([Atom('H', [0, 0, 0]), Atom('H', [0, 0, 1.4])])
print(h2)

In [ ]:
# Compute one- and two-electron integrals
integrals, basis_to_atom = compute_integrals(h2)

print(f"Number of basis functions: {integrals['S'].shape[0]}")
print(f"\nOverlap matrix (S):")
print(integrals['S'])

In [ ]:
# Run RHF calculation
rhf = RHF(h2, integrals)
energy = rhf.run(use_diis=True)

print(f"SCF converged in {rhf.scf_iterations} iterations")
print(f"Total energy: {energy:.6f} Hartree")
print(f"Expected (literature): -1.116700 Hartree")

## 2. Energy Analysis and Decomposition

Let's break down the total energy into its components.

In [ ]:
# Get energy components
components = decompose_energy(rhf)
print_energy_decomposition(rhf)
print(f"\nVirial ratio (-V/T): {virial_ratio(rhf):.6f} (should be ~2.0)")

In [ ]:
# Visualize energy decomposition
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: All energy components
labels = ['Kinetic', 'Nuclear\nAttraction', 'Two-electron', 'Nuclear\nRepulsion']
values = [
    components['kinetic'],
    components['nuclear_attraction'],
    components['two_electron'],
    components['nuclear_repulsion']
]
colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']

bars = ax1.bar(labels, values, color=colors, edgecolor='black', linewidth=1.2)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_ylabel('Energy (Hartree)')
ax1.set_title('Energy Component Decomposition')

for bar, val in zip(bars, values):
    height = bar.get_height()
    ax1.annotate(f'{val:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3 if height > 0 else -15),
                textcoords="offset points",
                ha='center', va='bottom' if height > 0 else 'top',
                fontsize=10)

# Right plot: Energy balance
balance_labels = ['One-electron', 'Two-electron', 'Nuclear Rep.', 'Total']
balance_values = [
    components['one_electron'],
    components['two_electron'],
    components['nuclear_repulsion'],
    components['total']
]
balance_colors = ['#3498db', '#e74c3c', '#9b59b6', '#2c3e50']

bars2 = ax2.bar(balance_labels, balance_values, color=balance_colors, edgecolor='black', linewidth=1.2)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Energy (Hartree)')
ax2.set_title('Energy Balance')

for bar, val in zip(bars2, balance_values):
    height = bar.get_height()
    ax2.annotate(f'{val:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3 if height > 0 else -15),
                textcoords="offset points",
                ha='center', va='bottom' if height > 0 else 'top',
                fontsize=10)

plt.tight_layout()
plt.show()

## 3. Population Analysis

Mulliken population analysis provides atomic charges and bond orders.

In [ ]:
# Perform Mulliken analysis
mulliken = MullikenPopulation.compute(rhf, basis_on_atoms=[0, 1])
MullikenPopulation.print_analysis(rhf, basis_on_atoms=[0, 1])

In [ ]:
# Visualize population analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Atomic populations
atoms = ['H1', 'H2']
populations = mulliken['gross_populations']
charges = mulliken['charges']

x = np.arange(len(atoms))
width = 0.35

bars1 = ax1.bar(x - width/2, populations, width, label='Population', color='#3498db')
bars2 = ax1.bar(x + width/2, charges, width, label='Charge', color='#e74c3c')

ax1.set_ylabel('Electrons')
ax1.set_title('Mulliken Population Analysis')
ax1.set_xticks(x)
ax1.set_xticklabels(atoms)
ax1.legend()
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Bond orders
bond_order = mulliken['bond_orders'][0, 1]
ax2.bar(['H1-H2'], [bond_order], color='#2ecc71', edgecolor='black', linewidth=1.2)
ax2.set_ylabel('Bond Order')
ax2.set_title('Mulliken Bond Order')
ax2.set_ylim(0, 1.2)
ax2.annotate(f'{bond_order:.4f}', xy=(0, bond_order), xytext=(0, 5),
            textcoords="offset points", ha='center', fontsize=12)

plt.tight_layout()
plt.show()

## 4. Molecular Orbital Analysis

Koopmans' theorem relates orbital energies to ionization potentials and electron affinities.

In [ ]:
# Get molecular orbital information
mo = rhf.get_molecular_orbitals()
print_koopmans_analysis(rhf)

print("\nMolecular Orbital Coefficients:")
for i in range(rhf.n_basis):
    occ = "occ" if mo['occupations'][i] > 0 else "vir"
    coeffs = mo['coefficients'][:, i]
    print(f"MO {i+1} ({occ}): E = {mo['energies'][i]:.6f} Ha, coeffs = [{coeffs[0]:.4f}, {coeffs[1]:.4f}]")

In [ ]:
# Visualize molecular orbital energy diagram
fig, ax = plt.subplots(figsize=(8, 6))

energies = mo['energies']
occupations = mo['occupations']
n_occ = rhf.n_occ

# Draw orbital energy levels
for i, (e, occ) in enumerate(zip(energies, occupations)):
    color = '#3498db' if occ > 0 else '#e74c3c'
    label = 'Occupied' if i == 0 else ('Virtual' if i == n_occ else None)
    ax.hlines(e, 0.3, 0.7, colors=color, linewidth=4, label=label)
    
    # Add orbital label
    orbital_name = 'HOMO' if i == n_occ - 1 else ('LUMO' if i == n_occ else f'MO{i+1}')
    ax.text(0.75, e, f'{orbital_name}: {e:.4f} Ha ({e*HARTREE_TO_EV:.2f} eV)',
           va='center', fontsize=10)
    
    # Draw electrons
    if occ > 0:
        ax.annotate('', xy=(0.45, e+0.02), xytext=(0.45, e-0.02),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))
        ax.annotate('', xy=(0.55, e-0.02), xytext=(0.55, e+0.02),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Draw HOMO-LUMO gap
gap = homo_lumo_gap(rhf)
homo_e = energies[n_occ - 1]
lumo_e = energies[n_occ]
ax.annotate('', xy=(0.15, lumo_e), xytext=(0.15, homo_e),
           arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(0.05, (homo_e + lumo_e)/2, f'Gap\n{gap:.4f} Ha\n({gap*HARTREE_TO_EV:.2f} eV)',
       va='center', ha='center', fontsize=9, color='green')

ax.set_xlim(0, 1.5)
ax.set_ylabel('Energy (Hartree)')
ax.set_title('Molecular Orbital Energy Diagram for H$_2$')
ax.set_xticks([])
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 5. DIIS Convergence Acceleration

Direct Inversion in the Iterative Subspace (DIIS) significantly accelerates SCF convergence.

In [ ]:
# Compare convergence with and without DIIS
h2 = Molecule([Atom('H', [0, 0, 0]), Atom('H', [0, 0, 1.4])])
integrals, _ = compute_integrals(h2)

# Without DIIS
rhf_no_diis = RHF(h2, integrals)
energy_no_diis = rhf_no_diis.run(use_diis=False)

# With DIIS
rhf_with_diis = RHF(h2, integrals)
energy_with_diis = rhf_with_diis.run(use_diis=True)

print(f"Without DIIS: {rhf_no_diis.scf_iterations} iterations, E = {energy_no_diis:.10f} Ha")
print(f"With DIIS:    {rhf_with_diis.scf_iterations} iterations, E = {energy_with_diis:.10f} Ha")
print(f"Speedup: {rhf_no_diis.scf_iterations / rhf_with_diis.scf_iterations:.1f}x")

In [ ]:
# Visualize DIIS comparison
fig, ax = plt.subplots(figsize=(8, 5))

methods = ['Without DIIS', 'With DIIS']
iterations = [rhf_no_diis.scf_iterations, rhf_with_diis.scf_iterations]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, iterations, color=colors, edgecolor='black', linewidth=1.5)

for bar, iters in zip(bars, iterations):
    ax.annotate(f'{iters}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
               xytext=(0, 5), textcoords='offset points', ha='center', fontsize=14, fontweight='bold')

ax.set_ylabel('SCF Iterations')
ax.set_title('Effect of DIIS on SCF Convergence')
ax.set_ylim(0, max(iterations) * 1.2)

plt.tight_layout()
plt.show()

## 6. Potential Energy Surface Scan

Let's compute how the energy varies with the H-H bond distance.

In [ ]:
# Scan bond distances from 0.8 to 5.0 Bohr
distances = np.linspace(0.8, 5.0, 30)
energies = []
kinetic_energies = []
potential_energies = []

for r in distances:
    mol = Molecule([Atom('H', [0, 0, 0]), Atom('H', [0, 0, r])])
    ints, _ = compute_integrals(mol)
    rhf = RHF(mol, ints)
    try:
        rhf.run(use_diis=True)
        energies.append(rhf.energy)
        comp = decompose_energy(rhf)
        kinetic_energies.append(comp['kinetic'])
        potential_energies.append(comp['nuclear_attraction'] + comp['two_electron'] + comp['nuclear_repulsion'])
    except RuntimeError:
        energies.append(np.nan)
        kinetic_energies.append(np.nan)
        potential_energies.append(np.nan)

energies = np.array(energies)
kinetic_energies = np.array(kinetic_energies)
potential_energies = np.array(potential_energies)

In [ ]:
# Find equilibrium geometry
valid_mask = ~np.isnan(energies)
min_idx = np.nanargmin(energies)
r_eq = distances[min_idx]
e_min = energies[min_idx]

print(f"Equilibrium bond length: {r_eq:.3f} Bohr ({r_eq * BOHR_TO_ANGSTROM:.3f} Angstrom)")
print(f"Minimum energy: {e_min:.6f} Hartree")

In [ ]:
# Plot potential energy surface
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Total energy curve
ax1.plot(distances[valid_mask], energies[valid_mask], 'b-', linewidth=2, label='Total Energy')
ax1.axvline(x=r_eq, color='red', linestyle='--', alpha=0.7, label=f'$r_{{eq}}$ = {r_eq:.2f} Bohr')
ax1.axhline(y=e_min, color='gray', linestyle=':', alpha=0.5)
ax1.scatter([r_eq], [e_min], color='red', s=100, zorder=5)

ax1.set_xlabel('H-H Distance (Bohr)')
ax1.set_ylabel('Energy (Hartree)')
ax1.set_title('H$_2$ Potential Energy Curve')
ax1.legend()

# Right: Energy components
ax2.plot(distances[valid_mask], kinetic_energies[valid_mask], 'g-', linewidth=2, label='Kinetic (T)')
ax2.plot(distances[valid_mask], potential_energies[valid_mask], 'r-', linewidth=2, label='Potential (V)')
ax2.plot(distances[valid_mask], energies[valid_mask], 'b--', linewidth=2, label='Total (E)')
ax2.axvline(x=r_eq, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

ax2.set_xlabel('H-H Distance (Bohr)')
ax2.set_ylabel('Energy (Hartree)')
ax2.set_title('Energy Components vs Bond Distance')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Comparing Different Molecules

Let's compare properties across different diatomic molecules.

In [ ]:
# Define molecules to compare
molecules_data = {
    'H2': {'atoms': [('H', 0), ('H', 1.4)], 'basis_map': [0, 1]},
    'HeH+': {'atoms': [('He', 0), ('H', 1.5)], 'basis_map': [0, 1], 'charge': 1},
    'LiH': {'atoms': [('Li', 0), ('H', 3.0)], 'basis_map': [0, 0, 0, 0, 0, 1]},
}

results = {}

for name, data in molecules_data.items():
    atoms = [Atom(sym, [0, 0, z]) for sym, z in data['atoms']]
    charge = data.get('charge', 0)
    mol = Molecule(atoms, charge=charge)
    
    ints, _ = compute_integrals(mol)
    rhf = RHF(mol, ints)
    
    try:
        rhf.run(use_diis=True)
        comp = decompose_energy(rhf)
        
        results[name] = {
            'energy': rhf.energy,
            'iterations': rhf.scf_iterations,
            'n_electrons': mol.n_electrons,
            'virial': virial_ratio(rhf),
            'homo': rhf.orbital_energies[rhf.n_occ - 1],
            'gap': homo_lumo_gap(rhf) if rhf.n_basis > rhf.n_occ else None
        }
        print(f"{name}: E = {rhf.energy:.6f} Ha, {rhf.scf_iterations} iterations")
    except Exception as e:
        print(f"{name}: Failed - {e}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

names = list(results.keys())
colors = ['#3498db', '#2ecc71', '#e74c3c']

# Total energies
ax = axes[0, 0]
energies_list = [results[n]['energy'] for n in names]
bars = ax.bar(names, energies_list, color=colors, edgecolor='black')
ax.set_ylabel('Total Energy (Hartree)')
ax.set_title('Total Energy Comparison')
for bar, e in zip(bars, energies_list):
    ax.annotate(f'{e:.4f}', xy=(bar.get_x() + bar.get_width()/2, e),
               xytext=(0, -15), textcoords='offset points', ha='center', fontsize=10)

# HOMO energies
ax = axes[0, 1]
homos = [results[n]['homo'] for n in names]
homos_ev = [h * HARTREE_TO_EV for h in homos]
bars = ax.bar(names, homos_ev, color=colors, edgecolor='black')
ax.set_ylabel('HOMO Energy (eV)')
ax.set_title('HOMO Energy (Ionization Potential)')
for bar, e in zip(bars, homos_ev):
    ax.annotate(f'{e:.2f}', xy=(bar.get_x() + bar.get_width()/2, e),
               xytext=(0, -15), textcoords='offset points', ha='center', fontsize=10)

# HOMO-LUMO gaps
ax = axes[1, 0]
gaps = [results[n]['gap'] * HARTREE_TO_EV if results[n]['gap'] else 0 for n in names]
bars = ax.bar(names, gaps, color=colors, edgecolor='black')
ax.set_ylabel('HOMO-LUMO Gap (eV)')
ax.set_title('HOMO-LUMO Gap')
for bar, g in zip(bars, gaps):
    ax.annotate(f'{g:.2f}', xy=(bar.get_x() + bar.get_width()/2, g),
               xytext=(0, 5), textcoords='offset points', ha='center', fontsize=10)

# Virial ratios
ax = axes[1, 1]
virials = [results[n]['virial'] for n in names]
bars = ax.bar(names, virials, color=colors, edgecolor='black')
ax.axhline(y=2.0, color='red', linestyle='--', label='Ideal = 2.0')
ax.set_ylabel('Virial Ratio (-V/T)')
ax.set_title('Virial Theorem Check')
ax.legend()
ax.set_ylim(1.8, 2.2)
for bar, v in zip(bars, virials):
    ax.annotate(f'{v:.4f}', xy=(bar.get_x() + bar.get_width()/2, v),
               xytext=(0, 5), textcoords='offset points', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## Summary

This tutorial demonstrated the key features of PyHF:

- **Molecule construction** with the `Atom` and `Molecule` classes
- **Integral computation** using STO-3G basis sets
- **RHF calculations** with optional DIIS acceleration
- **Energy decomposition** into kinetic, nuclear attraction, electron repulsion, and nuclear repulsion components
- **Population analysis** using Mulliken's method for atomic charges and bond orders
- **Molecular orbital analysis** with Koopmans' theorem for ionization energies
- **Potential energy surface scans** to find equilibrium geometries

The library provides a clean, educational implementation of Hartree-Fock theory suitable for learning quantum chemistry concepts.